# Crash!💥 Boom!💥 Bang! 💥🚗🚙💥🚗
**A SQL based exploratory data analysis of traffic accidents in Okinawa, Japan**
  
---   
# <span style="color:#0279f0;">Part 1: Data Quality & Integrity Check</span>
---


### **1.1 Getting to know the Main Dataset**
**Choice of Tool:**
- Pandas; Since the csv file is only around 50MB, pandas can handle the data quality checks efficiently.

**Tasks:**
- Load main data set
- Retrieve column names and data types
- Count total number of records
- Check if there are any null values

##### 👩🏻‍💻  *<span style="color:#0279f0">Loading Main Dataset, Null Check & Record Count...</span>*

In [1]:
import pandas as pd

import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from notebooks.config import DATA_SRC, FILTERED_DATA_PATH

# Read the CSV directly into pandas data frame
df = pd.read_csv(DATA_SRC)

# Count total no. of records in dataset
print("--------------------")
print(f"✅ Data loaded successfully.")
print(f"✅ Total records: {len(df)}")
print("--------------------")

# Column information
col_info = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.count().values
})
print("Column Information:")
print(col_info.to_string(index=False))
print("--------------------")

#Quick validation: Show first few rows
print("First 5 rows:")
print(df.head())

--------------------
✅ Data loaded successfully.
✅ Total records: 290895
--------------------
Column Information:
                         Column Name Data Type  Non-Null Count
                       data_category     int64          290895
                     prefecture_code     int64          290895
                 police_station_code     int64          290895
                           report_id     int64          290895
               accident_details_code     int64          290895
                          fatalities     int64          290895
                     injured_persons     int64          290895
                          route_code     int64          290895
                       location_code     int64          290895
                   municipality_code     int64          290895
                      occurence_year     int64          290895
                     occurence_month     int64          290895
                      occurence_date     int64          290895
    

Total no. of records = 290895.  
Total no. of non-null records for each field = 290895.  
 **Great! No Null Records Found!**  
#### <span style="color:orange">Null Check Results: Passed ✅</span>
..........



### **1.2 Validate Dates and Prefecture (City) Codes**   

**Tasks:**
- Ensure the date range of the records are valid for the EDA (year 2024 only)
- Ensure the data is Okinawa-centric and does not contain data for other prefectures
- Okinawa's prefecture_code is 97 (refer to prefecture code documentation)

##### 👩🏻‍💻 *<span style="color:#0279f0">Validating date range and prefecture code...</span>*

In [2]:
# ========================================
#Check date ranges found in the data set
# ========================================

#Count no, of unique dates in entire data set
unique_days = df[['occurence_year', 'occurence_month', 'occurence_date']].drop_duplicates().shape[0]

#Get date ranges
results_date_validation = pd.DataFrame({
    'earliest_year': [df['occurence_year'].min()],
    'latest_year': [df['occurence_year'].max()],
    'earliest_month': [df['occurence_month'].min()],
    'latest_month': [df['occurence_month'].max()],
    'earliest_date': [df['occurence_date'].min()],
    'latest_date': [df['occurence_date'].max()],
    'unique_days': [unique_days]
})

#print results
print("\n\n----------------------------------------")
print("📆 DATE VALIDATION RESULTS")
print("----------------------------------------")
display(results_date_validation)




----------------------------------------
📆 DATE VALIDATION RESULTS
----------------------------------------


,earliest_year,latest_year,earliest_month,latest_month,earliest_date,latest_date,unique_days
0,2019,2024,1,12,1,31,709


The 2024 release contains earlier occurrence years. This analysis uses the recorded occurrence date. The next cell checks whether the recorded weekdays support the earlier dates and whether a year change could explain the June 13 gap.

In [3]:
from datetime import date

display(df.groupby('occurence_year').size().rename('national_records'))
okinawa = df.loc[df['prefecture_code'].eq(97)]
excluded = okinawa.loc[okinawa['occurence_year'].ne(2024)].copy()
dates = excluded[['occurence_year', 'occurence_month', 'occurence_date']].copy()
dates.columns = ['year', 'month', 'day']
recorded = pd.to_datetime(dates)
as_2024 = pd.to_datetime(dates.assign(year=2024), errors='coerce')
weekday_code = excluded['occurence_day_code']

audit = excluded.assign(
    weekday_matches=((recorded.dt.dayofweek + 1) % 7 + 1).eq(weekday_code),
    weekday_matches_if_2024=((as_2024.dt.dayofweek + 1) % 7 + 1).eq(weekday_code),
    june_13=dates['month'].eq(6) & dates['day'].eq(13),
)
display(audit.groupby('occurence_year').agg(
    records=('report_id', 'size'),
    matching_weekdays=('weekday_matches', 'sum'),
    matching_weekdays_if_2024=('weekday_matches_if_2024', 'sum'),
    june_13_records=('june_13', 'sum'),
))

occurence_year
2019         1
2020         1
2021         3
2022        37
2023      8477
2024    282376
Name: national_records, dtype: int64

,records,matching_weekdays,matching_weekdays_if_2024,june_13_records
occurence_year,,,,
2022,3,3,0,0
2023,123,123,0,0


### 📊 Earlier dates in the 2024 release

All 126 excluded Okinawa records have weekdays consistent with their recorded earlier dates. None matches after changing only the year to 2024, and none has a June 13 month and day. This supports retaining the recorded years.

The full release contains 2,875 Okinawa records, matching the [Okinawa Police year-end tally](https://www.police.pref.okinawa.jp/docs/2024022700036/file_contents/R6_zinnsinnziko_12.pdf). The notebooks examine the 2,749 records whose occurrence year is 2024.

### **1.3 Final Data Preparation Step**   

**Tasks:**
- Filter out 2024 Okinawa specific data
- Identify the primary key
- Export subset data as csv for analysis

##### 👩🏻‍💻 *<span style="color:#0279f0">Filter, Identify, Export...</span>*

In [4]:
okinawa_2024 = okinawa.loc[okinawa['occurence_year'].eq(2024)].copy()

dates = okinawa_2024[['occurence_year', 'occurence_month', 'occurence_date']].copy()
dates.columns = ['year', 'month', 'day']
observed = pd.DatetimeIndex(pd.to_datetime(dates).unique())
calendar = pd.date_range(date(2024, 1, 1), date(2024, 12, 31))
print(f"{len(okinawa_2024):,} Okinawa crashes with 2024 occurrence dates")
print("Dates with no records:", calendar.difference(observed).strftime("%Y-%m-%d").tolist())

key = ['report_id', 'police_station_code']
assert okinawa_2024[key].notna().all().all(), "Missing Okinawa crash key"
assert not okinawa_2024.duplicated(key).any(), "Duplicate Okinawa crash key"

national_key = ['prefecture_code', *key]
assert df[national_key].notna().all().all(), "Missing national crash key"
assert not df.duplicated(national_key).any(), "Duplicate national crash key"

2,749 Okinawa crashes with 2024 occurrence dates
Dates with no records: ['2024-06-13']


### ✅ Summary of the Okinawa subset

The 2,749 rows cover 365 dates. The file has no Okinawa row for June 13, so it cannot establish whether that day had zero crashes or records are missing. The Okinawa key is the report ID and police station code. The national key also includes the prefecture code.

👉🏼 Note: *The export cell below will only run if EXPORT_COMPLETE = False. This prevents accidental overwrites. To re-export the CSV (e.g., after filtering changes), change the flag to False and run the cell.*  

In [5]:
# ========================================
# Note: The export cell below will only run if EXPORT_COMPLETE = False. 
# This prevents accidental overwrites. 
# To re-export the CSV (e.g., after filtering changes), change the flag to False and run the cell.
# ========================================

# Set this flag to False if you need to re-export the CSV
EXPORT_COMPLETE = True


if not EXPORT_COMPLETE:
    # Define output path (adjust relative path as needed)
    output_path = FILTERED_DATA_PATH
    
    # Export the csv
    okinawa_2024.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ Exported {len(okinawa_2024)} records to {output_path}")
else:
    print("⏭️ Export skipped (EXPORT_COMPLETE flag is True). Set to False to re-export.")

⏭️ Export skipped (EXPORT_COMPLETE flag is True). Set to False to re-export.


---
## 📊 The 2024 Okinawa Traffic Accidents Dataset is now ready for Exploratory Data Analysis.
This concludes the data preparation stage. I'll see you in the next chapter!